In [1]:
import os, sys

os.chdir(r"C:\Users\DELL\Desktop\SkillRecommendation-System")
sys.path.insert(0, r"C:\Users\DELL\Desktop\SkillRecommendation-System\projects")

print("Working dir:", os.getcwd())
print("Python path includes projects/:", any("projects" in p for p in sys.path))

Working dir: C:\Users\DELL\Desktop\SkillRecommendation-System
Python path includes projects/: True


In [2]:
import pandas as pd
import numpy as np


def load_data(filepath: str) -> pd.DataFrame:
    required_columns = {"student_id", "level_id", "score", "time_spent_minutes", "passed"}
    df = pd.read_csv(filepath)
    missing = required_columns - set(df.columns)
    if missing:
        raise ValueError(f"CSV is missing required columns: {missing}")
    df["student_id"] = df["student_id"].astype(str)
    df["level_id"] = df["level_id"].astype(str)
    df["score"] = df["score"].astype(float)
    df["time_spent_minutes"] = df["time_spent_minutes"].astype(float)
    df["passed"] = df["passed"].astype(bool).astype(int)
    df = df.dropna(subset=["student_id", "level_id", "score"]).reset_index(drop=True)
    return df


def build_feature_matrix(df: pd.DataFrame) -> pd.DataFrame:
    pivot = (
        df.groupby(["student_id", "level_id"])["score"]
        .mean()
        .unstack(fill_value=0)
    )
    all_levels = sorted(df["level_id"].unique())
    pivot = pivot.reindex(columns=all_levels, fill_value=0)
    return pivot


def get_completed_levels(df: pd.DataFrame, student_id: str) -> list:
    student_rows = df[df["student_id"] == student_id]
    return sorted(student_rows["level_id"].unique().tolist())


def get_level_pass_rates(df: pd.DataFrame) -> pd.Series:
    return df.groupby("level_id")["passed"].mean()


# --- Run ---
df = load_data("datasets/student_progress.csv")
feature_matrix = build_feature_matrix(df)
pass_rates = get_level_pass_rates(df)

print("=== Data Summary ===")
print(f"Total records   : {len(df)}")
print(f"Unique students : {df['student_id'].nunique()}")
print(f"Unique levels   : {df['level_id'].nunique()}")
print(f"\nFeature matrix shape: {feature_matrix.shape}")
print(f"\nPass rates per level:\n{pass_rates.round(2)}")

=== Data Summary ===
Total records   : 7890
Unique students : 1000
Unique levels   : 20

Feature matrix shape: (1000, 20)

Pass rates per level:
level_id
algo_dp                0.66
algo_greedy            0.67
algo_searching         0.68
algo_sorting           0.63
ds_dicts               0.58
ds_graphs              0.58
ds_lists               0.61
ds_trees               0.57
math_calculus          0.78
math_linear_algebra    0.69
math_probability       0.65
math_statistics        0.85
ml_classification      0.77
ml_clustering          0.75
ml_neural_nets         0.72
ml_regression          0.78
py_functions           0.60
py_loops               0.61
py_oop                 0.60
py_variables           0.58
Name: passed, dtype: float64
